# Spatial Integration

This notebook links cleaned EV charger records to ABS SA4 regions, investigates unmatched records and exports tables for database integration. Charger coordinates are processed under an explicit, provisional WGS84 assumption.


## 1. Objective, Inputs and Outputs

### Objective

Associate each cleaned EV charger source record with its containing Statistical Area Level 4 (SA4) region using its coordinates. Preserve the source record identifiers and existing quality flags, and provide regional attributes and matching status for downstream database integration.

### Inputs

- `data/processed/chargers_clean.csv`: the cleaned dataset produced by the acquisition and cleaning notebook from `ev_20251216.csv` (December 2025). The current input contains 1,958 source records. Spatial processing uses `record_id`, `Longitude`, and `Latitude`.
- `data/raw/sa4_2026/SA4_2026_AUST_GDA2020.shp` and its companion files: ABS ASGS Edition 4, 2026 SA4 boundaries for Australia. The boundary CRS is GDA2020 (EPSG:7844), as recorded in the `.prj` file.

Paths are relative to the project root. Run the acquisition and cleaning notebook first, or provide its corresponding outputs in these locations. The charger CRS is not confirmed by the source documentation; this notebook uses WGS84 (EPSG:4326) as an explicit, provisional assumption.

### Outputs

- `data/processed/charger_sa4.csv`: the main handoff table, retaining the input attributes and adding `SA4_CODE26`, `SA4_NAME26`, `STE_NAME26`, matching status, and CRS/version information. Records without a spatial match remain in the table with missing SA4 attributes.
- `data/processed/charger_sa4_review.csv`: a separate review table containing unmatched record identifiers, addresses, nearest candidate SA4 codes and names, candidate distances in metres, and review/CRS information. Candidate regions are diagnostic suggestions, not confirmed assignments.

### Record Granularity and Handoff

The main output contains one row per input source record, identified by the unchanged `record_id`. A source record is not necessarily a unique physical charging site; records sharing coordinates are not automatically merged or removed. The database integration workflow should use `record_id` to link this output to the other project tables and read identifiers and SA4 codes as text. Matching results remain conditional on the provisional charger CRS assumption.


## 2. Environment and Input Checks

Inspect input files, record identifiers, coordinate values and boundary geometries before spatial integration. Record the distinction between the verified boundary CRS and the assumed charger CRS.


### 2.1 Project Paths and Input Files

Locate the project root when running from either the root or the notebooks directory. Define the charger CSV and SA4 shapefile paths and report whether both files exist.


In [16]:
from pathlib import Path

working_dir = Path.cwd().resolve()

if (working_dir / "data").is_dir():
    PROJECT_ROOT = working_dir
elif working_dir.name == "notebooks" and (working_dir.parent / "data").is_dir():
    PROJECT_ROOT = working_dir.parent
else:
    raise FileNotFoundError(
        "Open the project root or its notebooks directory before running."
    )

CHARGERS_PATH = PROJECT_ROOT / "data" / "processed" / "chargers_clean.csv"
SA4_PATH = (
    PROJECT_ROOT / "data" / "raw" / "sa4_2026"
    / "SA4_2026_AUST_GDA2020.shp"
)

print("Charger file exists:", CHARGERS_PATH.is_file())
print("SA4 boundary file exists:", SA4_PATH.is_file())

Charger file exists: True
SA4 boundary file exists: True


### 2.2 Charger Records and Identifiers

Read the cleaned charger CSV, preserving record_id as text. Inspect the record count, column names, missing identifiers and duplicate identifiers, then preview the data. These identifiers refer to source records, not necessarily distinct physical sites.


In [17]:
import pandas as pd

chargers = pd.read_csv(
    CHARGERS_PATH,
    dtype={
        "record_id": "string",
        "OBJECTID": "string",
        "PCODE": "string",
        "postcode_from_address": "string",
        "postcode_standardized": "string"
    }
)

print("Number of records:", len(chargers))
print("Column names:", chargers.columns.tolist())
print("Missing record IDs:", chargers["record_id"].isna().sum())
print("Duplicate record IDs:", chargers["record_id"].duplicated().sum())

chargers.head()

Number of records: 1958
Column names: ['record_id', 'OBJECTID', 'Station_name', 'Station_address', 'Operator', 'Number_of_plugs', 'Charger_Type', 'Charger_rating', 'Latitude', 'Longitude', 'LGANAME', 'PCODE', 'Source', 'missing_region_metadata', 'power_kw', 'power_format', 'operator_raw', 'operator_standardized', 'operator_needs_review', 'source_file', 'shared_coordinates', 'shared_address_operator', 'potential_duplicate_review', 'postcode_from_address', 'postcode_check', 'postcode_standardized', 'postcode_source', 'postcode_needs_review', 'postcode_resolution', 'charger_type_standardized', 'status_from_type']
Missing record IDs: 0
Duplicate record IDs: 0


,record_id,OBJECTID,Station_name,Station_address,Operator,Number_of_plugs,Charger_Type,Charger_rating,Latitude,Longitude,...,shared_address_operator,potential_duplicate_review,postcode_from_address,postcode_check,postcode_standardized,postcode_source,postcode_needs_review,postcode_resolution,charger_type_standardized,status_from_type
0,ev_35fd7b517c13af2f3539c4e4bf763c10205c189b8da...,<NA>,NaN,", Muswellbrook, 2333",EVUp,2,AC,22 kW,-32.262242,150.890139,...,False,False,<NA>,no_address_candidate,2333,PCODE,False,existing_retained,AC,not_specified
1,ev_e795ae9310df33c9b9f8840aea501aec56db120b534...,<NA>,NaN,"01 Wallgrove Road, Sydney, 2766",BP,4,DC,150 kW,-33.811004,150.849597,...,False,False,<NA>,no_address_candidate,2766,PCODE,False,existing_retained,DC,not_specified
2,ev_92a54038955adbf16c92062ab57a4b497be32cb889f...,<NA>,NaN,"1 - 7 Ross St, Wilcannia NSW 2836, Australia",NRMA,4,DC,50 kW,-30.511874,151.669395,...,False,False,2836,conflicts_with_existing,<NA>,NaN,True,value_conflict,DC,not_specified
3,ev_3da79c501d7974c4467f521947a6948bf95b2e22c60...,<NA>,NaN,"1 Balfour St, Sydney, 2070",Chargefox,7,AC,22 kW,-33.774101,151.167035,...,False,False,<NA>,no_address_candidate,2070,PCODE,False,existing_retained,AC,not_specified
4,ev_1db50d53783f3bb52cca4274ed47e15067c953845d5...,<NA>,NaN,"1 Bay Ln, Byron Bay, 2481",Tesla,2,AC,19 kW,-28.641819,153.613633,...,False,False,<NA>,no_address_candidate,2481,PCODE,False,existing_retained,AC,not_specified


### 2.3 SA4 Boundary Data and CRS

Read the SA4 shapefile and inspect its attributes and coordinate reference system. Count missing and empty geometries separately to identify records that cannot be used for spatial matching.


In [18]:
import geopandas as gpd

sa4 = gpd.read_file(SA4_PATH)

print("Number of boundary records:", len(sa4))
print("Coordinate reference system:", sa4.crs)
print("Column names:", sa4.columns.tolist())
print("Missing geometries:", sa4.geometry.isna().sum())
print("Empty geometries:", sa4.geometry.is_empty.sum())

sa4.head()

Number of boundary records: 108
Coordinate reference system: EPSG:7844
Column names: ['SA4_CODE26', 'SA4_NAME26', 'CHG_FLAG26', 'CHG_LBL26', 'GCC_CODE26', 'GCC_NAME26', 'STE_CODE26', 'STE_NAME26', 'AUS_CODE26', 'AUS_NAME26', 'AREASQKM26', 'geometry']
Missing geometries: 19
Empty geometries: 0


,SA4_CODE26,SA4_NAME26,CHG_FLAG26,CHG_LBL26,GCC_CODE26,GCC_NAME26,STE_CODE26,STE_NAME26,AUS_CODE26,AUS_NAME26,AREASQKM26,geometry
0,101,Capital Region,0,No change,1RNSW,Rest of NSW,1,New South Wales,AUS,Australia,51896.2445,"MULTIPOLYGON (((150.08176 -36.377, 150.08159 -..."
1,102,Central Coast,0,No change,1GSYD,Greater Sydney,1,New South Wales,AUS,Australia,1681.0090,"MULTIPOLYGON (((151.44404 -33.49089, 151.44377..."
2,103,Central West,0,No change,1RNSW,Rest of NSW,1,New South Wales,AUS,Australia,70297.0605,"POLYGON ((150.17101 -33.78914, 150.17099 -33.7..."
3,104,Coffs Harbour - Grafton,0,No change,1RNSW,Rest of NSW,1,New South Wales,AUS,Australia,13229.7573,"MULTIPOLYGON (((153.3615 -29.35637, 153.36139 ..."
4,105,Far West and Orana,0,No change,1RNSW,Rest of NSW,1,New South Wales,AUS,Australia,339355.6495,"POLYGON ((149.21457 -32.82352, 149.21438 -32.8..."


### 2.4 Review of Missing Boundary Geometry

Display the codes, names and states of records with missing geometry. This allows non-spatial categories to be identified before excluding them from the spatial lookup; the original boundary dataset remains unchanged.


In [19]:
missing_geometry = sa4.loc[
    sa4.geometry.isna(),
    ["SA4_CODE26", "SA4_NAME26", "STE_CODE26", "STE_NAME26"]
]

print("Records with missing geometry:")
display(missing_geometry)

Records with missing geometry:


,SA4_CODE26,SA4_NAME26,STE_CODE26,STE_NAME26
28,197,Migratory - Offshore - Shipping (NSW),1,New South Wales
29,199,No usual address (NSW),1,New South Wales
47,297,Migratory - Offshore - Shipping (Vic),2,Victoria
48,299,No usual address (Vic),2,Victoria
68,397,Migratory - Offshore - Shipping (Qld),3,Queensland
69,399,No usual address (Qld),3,Queensland
77,497,Migratory - Offshore - Shipping (SA),4,South Australia
78,499,No usual address (SA),4,South Australia
89,597,Migratory - Offshore - Shipping (WA),5,Western Australia
90,599,No usual address (WA),5,Western Australia


### 2.5 Preparation and Validation of Boundary Geometry

Create a separate boundary dataset containing only non-missing, non-empty geometries. Report excluded records, geometric validity and geometry types. Both Polygon and MultiPolygon geometries can represent SA4 regions; invalid geometries are reported rather than automatically repaired.


In [20]:
# Keep only records with non-missing, non-empty geometry.
sa4_spatial = sa4.loc[
    sa4.geometry.notna() & ~sa4.geometry.is_empty
].copy()

print("Original boundary records:", len(sa4))
print("Boundary records with geometry:", len(sa4_spatial))
print("Excluded records:", len(sa4) - len(sa4_spatial))
print("Invalid geometries:", (~sa4_spatial.geometry.is_valid).sum())
print("Geometry types:")
print(sa4_spatial.geometry.geom_type.value_counts())

Original boundary records: 108
Boundary records with geometry: 89
Excluded records: 19
Invalid geometries: 0
Geometry types:
MultiPolygon    48
Polygon         41
Name: count, dtype: int64


### 2.6 Coordinate Values and Range Checks

Convert longitude and latitude to numeric values in a separate table and check their global ranges. Missing, non-numeric and out-of-range coordinates fail this check. Valid ranges do not establish the CRS or confirm that coordinates agree with station addresses.


In [21]:
coordinates = chargers[["Longitude", "Latitude"]].apply(
    pd.to_numeric, errors="coerce"
)

valid_coordinates = (
    coordinates["Longitude"].between(-180, 180)
    & coordinates["Latitude"].between(-90, 90)
)

print("Records with valid coordinate ranges:", valid_coordinates.sum())
print("Records with invalid or missing coordinates:", (~valid_coordinates).sum())
display(coordinates.describe())

Records with valid coordinate ranges: 1958
Records with invalid or missing coordinates: 0


,Longitude,Latitude
count,1958.000000,1958.000000
mean,150.697229,-33.484300
std,1.665905,1.588800
min,141.460104,-37.111463
25%,150.486904,-33.944959
50%,151.156083,-33.853122
75%,151.266295,-33.010212
max,153.615875,-28.168593


### 2.7 Provisional Charger CRS Assumption

Record WGS84 (EPSG:4326) as a working assumption for the charger coordinates because the reviewed source documentation does not specify their CRS. The boundary CRS is read from the shapefile. This step records the assumption without transforming coordinates, and subsequent results remain conditional on it.


In [22]:
# Working assumption; not confirmed by the source documentation.
CHARGER_CRS = "EPSG:4326"
CHARGER_CRS_STATUS = "Assumed WGS84; pending confirmation"

print("Assumed charger CRS:", CHARGER_CRS)
print("Charger CRS status:", CHARGER_CRS_STATUS)
print("Verified boundary CRS:", sa4_spatial.crs)

Assumed charger CRS: EPSG:4326
Charger CRS status: Assumed WGS84; pending confirmation
Verified boundary CRS: EPSG:7844


## 3. Spatial Integration

Construct charger point geometries, align coordinate reference systems and associate records with SA4 polygons.


### 3.1 Point Geometry and CRS Alignment

Create a point for each charger record using longitude as x and latitude as y, assigning the provisional charger CRS. Transform the point geometries to the boundary CRS and compare the resulting CRS values. Original coordinate columns are retained; only the geometry is transformed.


In [23]:
# Stop if any coordinates fail the input checks.
assert valid_coordinates.all(), "Review invalid coordinates before proceeding."

# Create points using longitude as x and latitude as y.
charger_points = gpd.GeoDataFrame(
    chargers.copy(),
    geometry=gpd.points_from_xy(
        coordinates["Longitude"],
        coordinates["Latitude"]
    ),
    crs=CHARGER_CRS
)

# Transform point geometry to the boundary CRS.
charger_points = charger_points.to_crs(sa4_spatial.crs)

print("Number of point records:", len(charger_points))
print("Point CRS:", charger_points.crs)
print("Boundary CRS:", sa4_spatial.crs)
print("CRS match:", charger_points.crs == sa4_spatial.crs)

Number of point records: 1958
Point CRS: EPSG:7844
Boundary CRS: EPSG:7844
CRS match: True


### 3.2 Point-in-Polygon Matching

Use a left spatial join with the within predicate to identify the SA4 polygon containing each point. Retain unmatched charger records and attach the SA4 code, name and state to matched records. Points exactly on polygon boundaries are excluded by within, and overlapping matches can produce additional rows.


In [24]:
# Select the boundary attributes needed for matching.
sa4_lookup = sa4_spatial[
    ["SA4_CODE26", "SA4_NAME26", "STE_NAME26", "geometry"]
].copy()

# Retain every charger record, including unmatched records.
charger_sa4_matches = gpd.sjoin(
    charger_points,
    sa4_lookup,
    how="left",
    predicate="within"
)

print("Input charger records:", len(charger_points))
print("Spatial join rows:", len(charger_sa4_matches))
print(
    "Rows without an SA4 match:",
    charger_sa4_matches["index_right"].isna().sum()
)

display(
    charger_sa4_matches[
        ["record_id", "SA4_CODE26", "SA4_NAME26", "STE_NAME26"]
    ].head()
)

Input charger records: 1958
Spatial join rows: 1958
Rows without an SA4 match: 1


,record_id,SA4_CODE26,SA4_NAME26,STE_NAME26
0,ev_35fd7b517c13af2f3539c4e4bf763c10205c189b8da...,106,Hunter Valley exc Newcastle,New South Wales
1,ev_e795ae9310df33c9b9f8840aea501aec56db120b534...,116,Sydney - Blacktown,New South Wales
2,ev_92a54038955adbf16c92062ab57a4b497be32cb889f...,110,New England and North West,New South Wales
3,ev_3da79c501d7974c4467f521947a6948bf95b2e22c60...,121,Sydney - North Sydney and Hornsby,New South Wales
4,ev_1db50d53783f3bb52cca4274ed47e15067c953845d5...,112,Richmond - Tweed,New South Wales


## 4. Validation and Exceptions

Review matching coverage and cardinality, investigate unmatched points and inspect matches outside NSW.


### 4.1 Match Counts and Unmatched Records

Count non-missing boundary matches for each source record and distinguish zero, one and multiple matches. Display the addresses and coordinates of unmatched records for investigation. This checks match cardinality without assuming that every successful spatial match identifies the true station location.


In [25]:
# Count matched boundary records for each input record.
match_counts = (
    charger_sa4_matches.groupby("record_id")["index_right"]
    .count()
    .reindex(chargers["record_id"], fill_value=0)
)

print("Unmatched records:", (match_counts == 0).sum())
print("Records with one match:", (match_counts == 1).sum())
print("Records with multiple matches:", (match_counts > 1).sum())

unmatched_records = charger_sa4_matches.loc[
    charger_sa4_matches["index_right"].isna()
]

display(
    unmatched_records[
        [
            "record_id",
            "Station_name",
            "Station_address",
            "Longitude",
            "Latitude",
            "LGANAME"
        ]
    ]
)

Unmatched records: 1
Records with one match: 1957
Records with multiple matches: 0


,record_id,Station_name,Station_address,Longitude,Latitude,LGANAME
1833,ev_cba86a64f9e7cfd83617245c590c47c767ecd3439cb...,NaN,1 Sandy Bay Rd\nClontarf NSW 2093\nAustralia,151.25284,-33.80467,Northern Beaches Council


### 4.2 Boundary Intersection Check

Recheck unmatched points with the intersects predicate, which includes polygon boundaries. Use the original point table to avoid retaining fields from the earlier join. This diagnostic result does not replace the main within-based output.


In [26]:
# Retrieve the original points to avoid carrying previous join columns.
unmatched_points = charger_points.loc[
    charger_points["record_id"].isin(unmatched_records["record_id"])
].copy()

# Check whether unmatched points intersect a polygon or its boundary.
boundary_check = gpd.sjoin(
    unmatched_points,
    sa4_lookup,
    how="left",
    predicate="intersects"
)

display(
    boundary_check[
        ["record_id", "Station_address", "SA4_CODE26", "SA4_NAME26"]
    ]
)

,record_id,Station_address,SA4_CODE26,SA4_NAME26
1833,ev_cba86a64f9e7cfd83617245c590c47c767ecd3439cb...,1 Sandy Bay Rd\nClontarf NSW 2093\nAustralia,NaN,NaN


### 4.3 Nearest-Region Distance Review

When unmatched points exist, estimate a local UTM CRS and project points and polygons before measuring distances in metres. Identify nearest candidate SA4 regions for diagnosis only; proximity does not confirm an assignment or establish the cause of a mismatch. If no unmatched records remain, skip distance estimation and create an empty candidate table with the fields required for export.


In [27]:
# Skip distance estimation when there are no unmatched points.
if unmatched_points.empty:
    distance_crs = None
    nearest_candidates = unmatched_points.copy()
    for column in ["SA4_CODE26", "SA4_NAME26", "STE_NAME26"]:
        nearest_candidates[column] = pd.Series(
            index=nearest_candidates.index, dtype="string"
        )
    nearest_candidates["index_right"] = pd.Series(
        index=nearest_candidates.index, dtype="float64"
    )
    nearest_candidates["distance_m"] = pd.Series(
        index=nearest_candidates.index, dtype="float64"
    )
    print("No unmatched records; nearest-region review skipped.")
else:
    # Estimate a local projected CRS for distance measurement.
    distance_crs = unmatched_points.estimate_utm_crs()

    unmatched_projected = unmatched_points.to_crs(distance_crs)
    boundaries_projected = sa4_lookup.to_crs(distance_crs)

    # Find nearest candidate regions for diagnosis only.
    nearest_candidates = gpd.sjoin_nearest(
        unmatched_projected,
        boundaries_projected,
        how="left",
        distance_col="distance_m"
    )

    print("Distance measurement CRS:", distance_crs)

display(
    nearest_candidates[
        [
            "record_id",
            "Station_address",
            "SA4_CODE26",
            "SA4_NAME26",
            "distance_m"
        ]
    ]
)

Distance measurement CRS: EPSG:32756


,record_id,Station_address,SA4_CODE26,SA4_NAME26,distance_m
1833,ev_cba86a64f9e7cfd83617245c590c47c767ecd3439cb...,1 Sandy Bay Rd\nClontarf NSW 2093\nAustralia,122,Sydney - Northern Beaches,1.765063


### 4.4 Matches Outside NSW

Inspect matched records whose associated state is not New South Wales. Keeping Australian boundary coverage makes possible out-of-state matches visible. An empty result means no matched records fall outside NSW; it does not verify address-coordinate agreement within NSW.


In [28]:
outside_nsw = charger_sa4_matches.loc[
    charger_sa4_matches["index_right"].notna()
    & charger_sa4_matches["STE_NAME26"].ne("New South Wales")
]

print("Matched records outside NSW:", len(outside_nsw))

display(
    outside_nsw[
        [
            "record_id",
            "Station_address",
            "SA4_CODE26",
            "SA4_NAME26",
            "STE_NAME26"
        ]
    ]
)

Matched records outside NSW: 0


,record_id,Station_address,SA4_CODE26,SA4_NAME26,STE_NAME26


## 5. Saving and Handoff

Prepare and export the main results and a separate review table for downstream database integration. Preserve record identifiers and disclose CRS assumptions.


### 5.1 Main Handoff Table

Check that the join preserves one row per source record and the complete set of record identifiers. Remove geometry and the internal join index for CSV export, and add match status, the provisional charger CRS and boundary version information. Retain existing input attributes and quality flags, with SA4 attributes left missing for unmatched records.


In [29]:
# Ensure that each source record has exactly one output row.
assert charger_sa4_matches["record_id"].is_unique
assert len(charger_sa4_matches) == len(chargers)
assert set(charger_sa4_matches["record_id"]) == set(chargers["record_id"])

# Remove spatial objects and internal join fields from the CSV output.
charger_sa4_output = pd.DataFrame(
    charger_sa4_matches.drop(columns=["geometry", "index_right"])
).copy()

# Record matching status and the CRS assumption.
charger_sa4_output["sa4_match_status"] = (
    charger_sa4_output["SA4_CODE26"].notna()
    .map({True: "matched_within", False: "unmatched"})
)
charger_sa4_output["charger_crs_assumed"] = CHARGER_CRS
charger_sa4_output["charger_crs_status"] = CHARGER_CRS_STATUS
charger_sa4_output["sa4_boundary_crs"] = str(sa4_spatial.crs)
charger_sa4_output["sa4_boundary_version"] = "ASGS Edition 4, SA4 2026"

print("Output records:", len(charger_sa4_output))
print("Match status counts:")
print(charger_sa4_output["sa4_match_status"].value_counts())

Output records: 1958
Match status counts:
sa4_match_status
matched_within    1957
unmatched            1
Name: count, dtype: int64


### 5.2 CSV Export and Read-Back Checks

Save the main output and a separate nearest-candidate review table under data/processed. Label candidates as unresolved and retain distance and CRS information. Reopen the main CSV to check record counts, identifier uniqueness, identifier preservation and missing SA4 counts. These checks validate export consistency, not the accuracy of the source locations; rerunning overwrites the two output CSV files.


In [30]:
OUTPUT_DIR = PROJECT_ROOT / "data" / "processed"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Save the main output without a DataFrame index.
output_path = OUTPUT_DIR / "charger_sa4.csv"
charger_sa4_output.to_csv(output_path, index=False)

# Save nearest candidates separately; these are not confirmed matches.
review_output = pd.DataFrame(
    nearest_candidates[
        [
            "record_id",
            "Station_address",
            "SA4_CODE26",
            "SA4_NAME26",
            "distance_m"
        ]
    ]
).rename(
    columns={
        "SA4_CODE26": "candidate_sa4_code",
        "SA4_NAME26": "candidate_sa4_name",
        "distance_m": "candidate_distance_m"
    }
)

review_output["review_status"] = "unresolved"
review_output["charger_crs_status"] = CHARGER_CRS_STATUS
review_output["distance_crs"] = str(distance_crs)

review_path = OUTPUT_DIR / "charger_sa4_review.csv"
review_output.to_csv(review_path, index=False)

# Reopen the main output and verify record preservation.
saved_output = pd.read_csv(
    output_path,
    dtype={"record_id": "string", "SA4_CODE26": "string"}
)

assert len(saved_output) == len(chargers)
assert saved_output["record_id"].is_unique
assert set(saved_output["record_id"]) == set(chargers["record_id"])
assert (
    saved_output["SA4_CODE26"].isna().sum()
    == (match_counts == 0).sum()
)

print("Main output saved:", output_path.relative_to(PROJECT_ROOT))
print("Review output saved:", review_path.relative_to(PROJECT_ROOT))
print("Saved output validation passed.")

Main output saved: data/processed/charger_sa4.csv
Review output saved: data/processed/charger_sa4_review.csv
Saved output validation passed.
